In [14]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix


# PATHS

FAISS_PATH = r"C:\Users\rmont\Downloads\proyecto3_chatbot_musical\proyecto3_chatbot_musical\data\embeddings_cache\indice_parrafos.faiss"
CHUNKS_PATH = r"C:\Users\rmont\Downloads\proyecto3_chatbot_musical\proyecto3_chatbot_musical\data\embeddings_cache\chunks_parrafos.pkl"
EMB_PATH = r"C:\Users\rmont\Downloads\proyecto3_chatbot_musical\proyecto3_chatbot_musical\data\embeddings_cache\embeddings_parrafos.npy"


# CARGA RAG

print("📂 Cargando RAG...")

indice = faiss.read_index(FAISS_PATH)

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

embeddings = np.load(EMB_PATH)

model = SentenceTransformer("intfloat/e5-base-v2")

print("✅ RAG cargado")


# SISTEMA RAG

def sistema_rag(pregunta, top_k=3):
    q_emb = model.encode([pregunta], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)

    scores, idx = indice.search(q_emb, top_k)

    resultados = []
    for i in range(top_k):
        chunk = chunks[idx[0][i]]
        resultados.append({
            "titulo": chunk[0],
            "artista": chunk[1],
            "texto": chunk[2],
            "score": float(1 - scores[0][i])
        })

    return resultados



# SISTEMA SIN RAG

def sistema_sin_rag(pregunta):
    for c in chunks:
        if pregunta.lower() in c[2].lower():
            return c
    return None



# DATASET

dataset = []

for i in range(500):
    chunk = chunks[i % len(chunks)]
    texto = chunk[2]

    palabras = texto.split()
    q = " ".join(palabras[:5]) if len(palabras) > 5 else texto

    dataset.append({
        "q": q,
        "expected": texto
    })



# EVALUACIÓN

def evaluar(modelo):
    y_true = []
    y_pred = []

    for item in dataset:
        q = item["q"]
        expected = item["expected"]

        if modelo == sistema_rag:
            respuesta = modelo(q)[0]["texto"]
        else:
            res = modelo(q)
            respuesta = res[2] if res else ""

        score = 1 if expected[:50] in respuesta else 0

        y_true.append(1)
        y_pred.append(score)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)

    return acc, f1, cm


# RESULTADOS

print("\n📊 Evaluando RAG...")
acc_rag, f1_rag, cm_rag = evaluar(sistema_rag)

print("\n📊 Evaluando SIN RAG...")
acc_no, f1_no, cm_no = evaluar(sistema_sin_rag)

print("\n========================")
print("RESULTADOS")
print("========================")

print(f"RAG     -> Accuracy: {acc_rag:.4f} | F1: {f1_rag:.4f}")
print(f"NO RAG  -> Accuracy: {acc_no:.4f} | F1: {f1_no:.4f}")

print("\nMatriz confusión RAG:\n", cm_rag)
print("\nMatriz confusión NO RAG:\n", cm_no)



# PREGUNTA


pregunta = "songs about fight "

print("\n" + "="*60)
print("🔎 CONSULTA MANUAL")
print("="*60)

resultados = sistema_rag(pregunta, top_k=3)

print(f"\n❓ Pregunta: {pregunta}\n")

for i, r in enumerate(resultados):
    print(f"🎵 Resultado {i+1}")
    print(f"   Canción: {r['titulo']}")
    print(f"   Artista: {r['artista']}")
    print(f"   Score: {r['score']:.4f}")
    print("-"*50)

📂 Cargando RAG...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9633.18it/s]


✅ RAG cargado

📊 Evaluando RAG...

📊 Evaluando SIN RAG...

RESULTADOS
RAG     -> Accuracy: 0.9100 | F1: 0.9529
NO RAG  -> Accuracy: 0.1780 | F1: 0.3022

Matriz confusión RAG:
 [[  0   0]
 [ 45 455]]

Matriz confusión NO RAG:
 [[  0   0]
 [411  89]]

🔎 CONSULTA MANUAL

❓ Pregunta: songs about fight 

🎵 Resultado 1
   Canción: the fight
   Artista: avenged sevenfold
   Score: 0.6616
--------------------------------------------------
🎵 Resultado 2
   Canción: the fight song
   Artista: marilyn manson
   Score: 0.6560
--------------------------------------------------
🎵 Resultado 3
   Canción: hero
   Artista: ministry
   Score: 0.6518
--------------------------------------------------
